# Final Three-Fish Compact Export

Run this notebook on the computer that holds the full `outputs/evaluation/v2a-RSNs/` artifacts. It extracts manuscript-sized tables and figures without copying the large row-level files or strict-fold checkpoints to another computer.

The notebook intentionally skips the largest row-level files by default:

- `strict_behavior_predictions.csv`
- `strict_causal_oof_embeddings.csv`
- `.strict_fold_checkpoints/`

It writes a compact export under `outputs/evaluation/manuscript_export/final_three_fish/` and creates a zip file that should be small enough to transfer.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import shutil
import sys
import zipfile
import tempfile

import numpy as np
import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "ica_denoising").exists():
            return candidate
        nested = candidate / "ica-denoising"
        if (nested / "pyproject.toml").exists() and (nested / "src" / "ica_denoising").exists():
            return nested
    raise RuntimeError(f"Could not find ica-denoising project root from {start}")


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

mpl_cache = Path(tempfile.gettempdir()) / "ica-denoising-matplotlib-cache"
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

try:
    display
except NameError:
    def display(obj):
        if hasattr(obj, "head"):
            print(obj.head(20).to_string())
        else:
            print(obj)

import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from ica_denoising.evaluation_runner import _primary_unit_effects

sns.set_theme(style="whitegrid", context="talk")
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
DATASETS = [
    {
        "key": "v2a-RSNs/220119_F2_run11_fluorescence",
        "recording": "220119_F2_run11",
        "fish_id": "fish_1",
        "label": "fish 1",
    },
    {
        "key": "v2a-RSNs/220127_F4_run2_fluorescence",
        "recording": "220127_F4_run2",
        "fish_id": "fish_2",
        "label": "fish 2",
    },
    {
        "key": "v2a-RSNs/220210_F2_run5_fluorescence",
        "recording": "220210_F2_run5",
        "fish_id": "fish_4",
        "label": "fish 4",
    },
]

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "evaluation"
V2A_ROOT = OUTPUT_ROOT / "v2a-RSNs"
EXPORT_ROOT = OUTPUT_ROOT / "manuscript_export" / "final_three_fish"
TABLE_DIR = EXPORT_ROOT / "tables"
FIGURE_DIR = EXPORT_ROOT / "figures"
PACKAGE_DIR = EXPORT_ROOT / "package"
for path in (TABLE_DIR, FIGURE_DIR, PACKAGE_DIR):
    path.mkdir(parents=True, exist_ok=True)

# Leave as None for automatic mode: completed recordings are included, incomplete
# recordings are skipped. To force a subset, set this to recording IDs, e.g.
# ["220119_F2_run11", "220127_F4_run2"].
RECORDINGS_TO_EXPORT: list[str] | None = ["220119_F2_run11", "220127_F4_run2"]
SKIP_INCOMPLETE_RECORDINGS = True
REQUIRE_RUN_MANIFEST = True
READ_OPTIONAL_DETAIL_TABLES = False
SHOW_ALL_CONFIGURED_AUDIT = False

# These files are sufficient for aggregate tables and manuscript figures.
MANUSCRIPT_TABLE_SPECS = {
    "trace": "strict_trace_preservation_metrics.csv",
    "behavior": "strict_behavior_fold_metrics.csv",
    "behavior_uncertainty": "strict_behavior_block_uncertainty.csv",
    "trace_uncertainty": "strict_trace_block_contributions.csv",
    "causal": "strict_causal_fold_metrics.csv",
    "causal_uncertainty": "strict_causal_block_contributions.csv",
    "causal_sufficiency": "strict_causal_sufficiency.csv",
    "causal_sufficiency_uncertainty": "strict_causal_sufficiency_block_contributions.csv",
    "artifact_probe": "strict_artifact_probe_metrics.csv",
    "latent_behavior_correlations": "strict_causal_oof_latent_behavior_correlations.csv",
    "nested_selection": "nested_selection.csv",
    "bpi": "bpi_ablation.csv",
}
OPTIONAL_DETAIL_TABLE_SPECS = {
    "bpi_component_scores": "bpi_component_scores.csv",
    "leakage_audit": "leakage_audit.csv",
    "component_selections": "component_selections.csv",
    "cluster_stability": "cluster_stability.csv",
    "temporal_dependence": "temporal_dependence_diagnostics.csv",
    "variant_metadata": "variant_metadata.csv",
}
TABLE_SPECS = dict(MANUSCRIPT_TABLE_SPECS)
if READ_OPTIONAL_DETAIL_TABLES:
    TABLE_SPECS.update(OPTIONAL_DETAIL_TABLE_SPECS)

COMPLETION_FILE_SPECS = dict(MANUSCRIPT_TABLE_SPECS)
if REQUIRE_RUN_MANIFEST:
    COMPLETION_FILE_SPECS["run_manifest"] = "run_manifest.json"

# These are intentionally never read into the compact export.
SKIPPED_LARGE_FILES = [
    "strict_behavior_predictions.csv",
    "strict_causal_oof_embeddings.csv",
]
SKIPPED_LARGE_DIRS = [
    ".strict_fold_checkpoints/",
]

print(f"Reading from:  {V2A_ROOT}")
print(f"Writing to:    {EXPORT_ROOT}")
print(f"Recordings requested:        {RECORDINGS_TO_EXPORT or 'all configured'}")
print(f"Automatic completion filter: {SKIP_INCOMPLETE_RECORDINGS}")
print(f"Require run manifest:        {REQUIRE_RUN_MANIFEST}")
print(f"Read optional detail tables: {READ_OPTIONAL_DETAIL_TABLES}")
print("Skipped large row-level files:")
for name in SKIPPED_LARGE_FILES + SKIPPED_LARGE_DIRS:
    print(f"- {name}")


## 1. Audit Available Files

This cell reports which required/optional inputs exist and how large they are. It does not read the huge prediction or embedding files.

In [ ]:
def file_size_mb(path: Path) -> float:
    return path.stat().st_size / 1_000_000 if path.exists() else 0.0


def requested_datasets() -> tuple[list[dict], pd.DataFrame]:
    if RECORDINGS_TO_EXPORT is None:
        return list(DATASETS), pd.DataFrame()
    requested = set(RECORDINGS_TO_EXPORT)
    selected = [item for item in DATASETS if item["recording"] in requested]
    unknown = sorted(requested - {item["recording"] for item in DATASETS})
    if unknown:
        raise ValueError(f"Unknown recording(s) in RECORDINGS_TO_EXPORT: {unknown}")
    skipped = [
        {**item, "reason": "not requested", "missing_required_files": ""}
        for item in DATASETS
        if item["recording"] not in requested
    ]
    return selected, pd.DataFrame(skipped)


def audit_file_specs() -> dict[str, str]:
    specs = dict(TABLE_SPECS)
    for label, filename in COMPLETION_FILE_SPECS.items():
        specs.setdefault(label, filename)
    return specs


def completion_status(recording: str, audit_frame: pd.DataFrame) -> tuple[bool, list[str]]:
    required = audit_frame[
        (audit_frame["recording"] == recording)
        & audit_frame["table"].isin(COMPLETION_FILE_SPECS.keys())
    ]
    missing = required.loc[~required["exists"], "filename"].astype(str).tolist()
    return len(missing) == 0, missing


def audit_datasets(items: list[dict]) -> pd.DataFrame:
    rows = []
    for item in items:
        recording_dir = V2A_ROOT / item["recording"]
        for label, filename in audit_file_specs().items():
            path = recording_dir / filename
            rows.append(
                {
                    **item,
                    "table": label,
                    "filename": filename,
                    "path": str(path.relative_to(PROJECT_ROOT)),
                    "exists": path.exists(),
                    "size_mb": file_size_mb(path),
                }
            )
        for filename in SKIPPED_LARGE_FILES:
            path = recording_dir / filename
            rows.append(
                {
                    **item,
                    "table": "skipped_large",
                    "filename": filename,
                    "path": str(path.relative_to(PROJECT_ROOT)),
                    "exists": path.exists(),
                    "size_mb": file_size_mb(path),
                }
            )
        for dirname in SKIPPED_LARGE_DIRS:
            path = recording_dir / dirname.rstrip("/")
            rows.append(
                {
                    **item,
                    "table": "skipped_large_dir",
                    "filename": dirname,
                    "path": str(path.relative_to(PROJECT_ROOT)),
                    "exists": path.exists(),
                    "size_mb": 0.0,
                }
            )
    return pd.DataFrame(rows)


def select_active_datasets(items: list[dict], audit_frame: pd.DataFrame) -> tuple[list[dict], pd.DataFrame]:
    active: list[dict] = []
    skipped: list[dict] = []
    for item in items:
        recording = item["recording"]
        is_complete, missing = completion_status(recording, audit_frame)
        if SKIP_INCOMPLETE_RECORDINGS and not is_complete:
            skipped.append(
                {
                    **item,
                    "reason": "incomplete",
                    "missing_required_files": "; ".join(missing),
                }
            )
            continue
        active.append(item)
    return active, pd.DataFrame(skipped)


REQUESTED_DATASETS, not_requested_recordings = requested_datasets()
audit = audit_datasets(REQUESTED_DATASETS)
all_configured_audit = audit_datasets(DATASETS) if SHOW_ALL_CONFIGURED_AUDIT else audit.copy()
ACTIVE_DATASETS, incomplete_recordings = select_active_datasets(REQUESTED_DATASETS, audit)
skipped_recordings = pd.concat(
    [frame for frame in [not_requested_recordings, incomplete_recordings] if not frame.empty],
    ignore_index=True,
) if (not not_requested_recordings.empty or not incomplete_recordings.empty) else pd.DataFrame()
active_recordings = {item["recording"] for item in ACTIVE_DATASETS}
active_audit = audit[audit["recording"].isin(active_recordings)].copy()

included_recordings = pd.DataFrame(ACTIVE_DATASETS)
included_recordings.to_csv(TABLE_DIR / "included_recordings.csv", index=False)
skipped_recordings.to_csv(TABLE_DIR / "skipped_recordings.csv", index=False)
all_configured_audit.to_csv(TABLE_DIR / "source_file_audit_all_configured.csv", index=False)
active_audit.to_csv(TABLE_DIR / "source_file_audit.csv", index=False)

if skipped_recordings.empty:
    print("No configured recordings were skipped.")
else:
    print("Skipped recordings:")
    display(skipped_recordings)

if included_recordings.empty:
    raise RuntimeError(
        "No completed recordings are available for export. Check source_file_audit.csv."
    )

missing_core = active_audit[
    active_audit["table"].isin(COMPLETION_FILE_SPECS.keys())
    & ~active_audit["exists"]
]
if not missing_core.empty:
    print("Missing required files in included recordings:")
    display(missing_core[["label", "filename", "path"]])
else:
    print("All required files are present for included recordings.")

print("Included recordings:")
display(included_recordings[["label", "recording", "fish_id"]])
print(f"Audited {len(audit)} source entries across {len(REQUESTED_DATASETS)} requested recording(s).")
print(f"Exporting {len(ACTIVE_DATASETS)} completed recording(s).")
print("Active source-file audit:")
display(active_audit.sort_values(["recording", "exists", "size_mb"], ascending=[True, True, False]).head(60))


## 2. Build Compact Recording Aggregates

This reads only the selected top-level CSV files, adds recording/fish metadata, and writes compact aggregate tables. It avoids the large row-level prediction and embedding files.

In [ ]:
def read_recording_table(item: dict, label: str, filename: str) -> pd.DataFrame:
    path = V2A_ROOT / item["recording"] / filename
    if not path.exists():
        return pd.DataFrame()
    if path.stat().st_size == 0:
        empty_source_files.append(
            {
                **item,
                "source_table": label,
                "filename": filename,
                "path": str(path.relative_to(PROJECT_ROOT)),
                "reason": "zero-byte CSV",
            }
        )
        return pd.DataFrame()
    try:
        frame = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        empty_source_files.append(
            {
                **item,
                "source_table": label,
                "filename": filename,
                "path": str(path.relative_to(PROJECT_ROOT)),
                "reason": "no columns to parse",
            }
        )
        return pd.DataFrame()
    if frame.empty:
        empty_source_files.append(
            {
                **item,
                "source_table": label,
                "filename": filename,
                "path": str(path.relative_to(PROJECT_ROOT)),
                "reason": "header-only or no rows",
            }
        )
        return pd.DataFrame()
    metadata = {
        "dataset_key": item["key"],
        "recording": item["recording"],
        "fish_id": item["fish_id"],
        "label": item["label"],
    }
    for column, value in reversed(metadata.items()):
        if column in frame:
            frame[column] = value
        else:
            frame.insert(0, column, value)
    frame.insert(0, "source_table", label)
    return frame


empty_source_files: list[dict] = []
aggregate_tables: dict[str, pd.DataFrame] = {}
for label, filename in TABLE_SPECS.items():
    frames = [read_recording_table(item, label, filename) for item in ACTIVE_DATASETS]
    frames = [frame for frame in frames if not frame.empty]
    table = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    aggregate_tables[label] = table
    out = TABLE_DIR / f"recording_{label}.csv"
    table.to_csv(out, index=False)
    print(f"{label:32s} {len(table):8d} rows -> {out.relative_to(PROJECT_ROOT)}")

empty_source_files_table = pd.DataFrame(empty_source_files)
empty_source_files_table.to_csv(TABLE_DIR / "empty_source_files.csv", index=False)
if not empty_source_files_table.empty:
    print("Skipped empty source CSVs:")
    display(empty_source_files_table[["label", "recording", "filename", "reason"]])

behavior = aggregate_tables["behavior"]
causal = aggregate_tables["causal"]
trace = aggregate_tables["trace"]
bpi = aggregate_tables["bpi"]

if behavior.empty or causal.empty or trace.empty:
    raise RuntimeError("Core aggregate tables are empty; check source_file_audit.csv and included_recordings.csv.")

recording_primary = _primary_unit_effects(behavior, causal, unit_col="recording")
fish_primary = _primary_unit_effects(behavior, causal, unit_col="fish_id")
recording_primary.to_csv(TABLE_DIR / "recording_primary_effects.csv", index=False)
fish_primary.to_csv(TABLE_DIR / "fish_primary_effects.csv", index=False)
print("Primary effects written.")
display(fish_primary.head(20))


## 3. Write Manuscript Summary Tables

These summaries are intentionally small. They are the files to transfer if you only need manuscript tables and plots.

In [ ]:
def summarize_numeric(
    frame: pd.DataFrame,
    group_cols: list[str],
    value_cols: list[str],
) -> pd.DataFrame:
    value_cols = [col for col in value_cols if col in frame.columns]
    group_cols = [col for col in group_cols if col in frame.columns]
    if frame.empty or not value_cols or not group_cols:
        return pd.DataFrame()
    rows = []
    grouped = frame.groupby(group_cols, dropna=False)
    for keys, group in grouped:
        keys = keys if isinstance(keys, tuple) else (keys,)
        base = dict(zip(group_cols, keys))
        for value_col in value_cols:
            values = pd.to_numeric(group[value_col], errors="coerce").dropna()
            if values.empty:
                continue
            rows.append(
                {
                    **base,
                    "metric": value_col,
                    "n": int(values.size),
                    "mean": float(values.mean()),
                    "median": float(values.median()),
                    "std": float(values.std(ddof=1)) if values.size > 1 else 0.0,
                    "q25": float(values.quantile(0.25)),
                    "q75": float(values.quantile(0.75)),
                    "min": float(values.min()),
                    "max": float(values.max()),
                }
            )
    return pd.DataFrame(rows)


behavior_primary = behavior.copy()
if "target_variant" in behavior_primary.columns:
    mask = behavior_primary["target_variant"].isin(["primary", "input_local"])
    if mask.any():
        behavior_primary = behavior_primary[mask].copy()

behavior_summary = summarize_numeric(
    behavior_primary,
    ["fish_id", "recording", "target", "task", "comparison", "train_version", "test_version"],
    ["pearson", "spearman", "rmse", "nrmse", "balanced_accuracy", "macro_f1", "roc_auc"],
)
causal_primary = causal[causal.get("transition_model", "linear") == "linear"].copy() if "transition_model" in causal.columns else causal.copy()
causal_summary = summarize_numeric(
    causal_primary,
    ["fish_id", "recording", "variant", "transition_model", "target_shift"],
    [
        "dynamic_mse_normalized",
        "dynamic_improvement_vs_persistence",
        "angle_pearson",
        "vigor_pearson",
        "bout_balanced_accuracy",
    ],
)
trace_summary = summarize_numeric(
    trace,
    ["fish_id", "recording", "family", "variant", "reference"],
    [
        "global_pearson",
        "neuron_pearson_mean",
        "nrmse_mean",
        "mean_abs_delta",
        "retained_energy_fraction",
        "spectral_power_retention",
        "low_frequency_power_fraction_delta_from_raw",
    ],
)
bpi_summary = summarize_numeric(
    bpi,
    ["fish_id", "recording", "method", "component_subset", "normalization", "weighting"],
    ["bpi"],
)

summaries = {
    "behavior_method_summary": behavior_summary,
    "causal_method_summary": causal_summary,
    "trace_preservation_summary": trace_summary,
    "bpi_summary": bpi_summary,
    "recording_primary_effects": recording_primary,
    "fish_primary_effects": fish_primary,
}

nested = aggregate_tables.get("nested_selection", pd.DataFrame())
if not nested.empty:
    nested_cols = [col for col in ["fish_id", "recording", "outer_fold", "policy", "selected_candidate", "selected_variant", "variant", "score"] if col in nested.columns]
    nested_compact = nested[nested_cols].copy() if nested_cols else nested.copy()
    summaries["nested_selection_compact"] = nested_compact

for name, table in summaries.items():
    out = TABLE_DIR / f"{name}.csv"
    table.to_csv(out, index=False)
    print(f"{name:32s} {len(table):8d} rows -> {out.relative_to(PROJECT_ROOT)}")

display(behavior_summary.head(20))

## 4. Create Manuscript Figures

These are compact overview figures. They are not intended to replace detailed supplementary audits, but they are enough to inspect the final three-fish pattern without moving the large source files.

In [ ]:
def save_current_figure(name: str) -> None:
    for suffix in ("pdf", "png"):
        out = FIGURE_DIR / f"{name}.{suffix}"
        plt.savefig(out, bbox_inches="tight", dpi=220)
    plt.show()


def method_order(values: pd.Series) -> list[str]:
    preferred = ["raw", "fastica", "infomax", "sobi", "jade"]
    present = list(dict.fromkeys(values.dropna().astype(str)))
    return [m for m in preferred if m in present] + sorted(m for m in present if m not in preferred)

# Behavior: tail vigor within-recording Pearson.
behavior_plot = behavior_primary[
    (behavior_primary["target"] == "tail_vigor")
    & (behavior_primary["comparison"] == "within")
].copy()
if not behavior_plot.empty and "pearson" in behavior_plot.columns:
    plt.figure(figsize=(9, 5))
    order = method_order(behavior_plot["test_version"])
    sns.boxplot(data=behavior_plot, x="test_version", y="pearson", order=order, color="white")
    sns.stripplot(data=behavior_plot, x="test_version", y="pearson", order=order, hue="fish_id", dodge=True, alpha=0.75)
    plt.xlabel("Trace variant")
    plt.ylabel("Tail vigor Pearson")
    plt.title("Behavior preservation: tail vigor")
    plt.legend(title="Fish", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_current_figure("behavior_tail_vigor_within_pearson")

# Behavior: bout state balanced accuracy.
bout_plot = behavior_primary[
    (behavior_primary["target"] == "bout_state")
    & (behavior_primary["comparison"] == "within")
].copy()
if not bout_plot.empty and "balanced_accuracy" in bout_plot.columns:
    plt.figure(figsize=(9, 5))
    order = method_order(bout_plot["test_version"])
    sns.boxplot(data=bout_plot, x="test_version", y="balanced_accuracy", order=order, color="white")
    sns.stripplot(data=bout_plot, x="test_version", y="balanced_accuracy", order=order, hue="fish_id", dodge=True, alpha=0.75)
    plt.xlabel("Trace variant")
    plt.ylabel("Bout balanced accuracy")
    plt.title("Behavior preservation: bout state")
    plt.legend(title="Fish", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_current_figure("behavior_bout_state_within_balanced_accuracy")

# Causal state prediction: lower normalized MSE is better.
if not causal_primary.empty and "dynamic_mse_normalized" in causal_primary.columns:
    plt.figure(figsize=(10, 5))
    order = method_order(causal_primary["variant"])
    sns.boxplot(data=causal_primary, x="variant", y="dynamic_mse_normalized", order=order, color="white")
    sns.stripplot(data=causal_primary, x="variant", y="dynamic_mse_normalized", order=order, hue="fish_id", dodge=True, alpha=0.75)
    plt.xlabel("Trace variant")
    plt.ylabel("Normalized dynamic MSE")
    plt.title("Causal-state prediction error")
    plt.xticks(rotation=30, ha="right")
    plt.legend(title="Fish", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_current_figure("causal_dynamic_mse_normalized")

# Trace preservation: global Pearson.
trace_plot = trace.copy()
if not trace_plot.empty and "global_pearson" in trace_plot.columns:
    plt.figure(figsize=(10, 5))
    order = method_order(trace_plot["variant"])
    sns.boxplot(data=trace_plot, x="variant", y="global_pearson", order=order, color="white")
    sns.stripplot(data=trace_plot, x="variant", y="global_pearson", order=order, hue="fish_id", dodge=True, alpha=0.7)
    plt.xlabel("Trace variant")
    plt.ylabel("Global Pearson vs reference")
    plt.title("Trace preservation")
    plt.xticks(rotation=30, ha="right")
    plt.legend(title="Fish", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_current_figure("trace_global_pearson")

# BPI ablation overview.
if not bpi.empty and "bpi" in bpi.columns:
    bpi_plot = bpi.copy()
    plt.figure(figsize=(10, 5))
    order = method_order(bpi_plot["method"])
    sns.boxplot(data=bpi_plot, x="method", y="bpi", order=order, color="white")
    sns.stripplot(data=bpi_plot, x="method", y="bpi", order=order, hue="fish_id", dodge=True, alpha=0.7)
    plt.xlabel("Method")
    plt.ylabel("BPI")
    plt.title("Behavior Preservation Index")
    plt.legend(title="Fish", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_current_figure("bpi_by_method")

print(f"Figures written to {FIGURE_DIR.relative_to(PROJECT_ROOT)}")

## 5. Package the Compact Export

The zip file is the artifact to transfer back to the main computer. It contains only the compact tables, figures, and a manifest of what was exported.

In [ ]:
def directory_size_mb(path: Path) -> float:
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / 1_000_000


manifest_rows = []
for path in sorted(EXPORT_ROOT.rglob("*")):
    if path.is_file() and path.suffix != ".zip":
        manifest_rows.append(
            {
                "path": str(path.relative_to(EXPORT_ROOT)),
                "size_mb": path.stat().st_size / 1_000_000,
            }
        )
manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(EXPORT_ROOT / "compact_export_manifest.csv", index=False)

zip_path = EXPORT_ROOT / "final_three_fish_compact_export.zip"
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, mode="w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(EXPORT_ROOT.rglob("*")):
        if path.is_file() and path != zip_path:
            zf.write(path, arcname=path.relative_to(EXPORT_ROOT))

print(f"Compact export directory: {EXPORT_ROOT}")
print(f"Directory size: {directory_size_mb(EXPORT_ROOT):.2f} MB")
print(f"Zip package: {zip_path}")
print(f"Zip size: {zip_path.stat().st_size / 1_000_000:.2f} MB")
display(manifest.sort_values("size_mb", ascending=False).head(30))